In [3]:
# Cell 1 
# ============================================================
# 01_data_ingestion.ipynb
# Olist Supply Chain Risk Intelligence
# ============================================================
# This notebook:
# 1. Loads all 9 Olist CSVs
# 2. Parses and cleans datetime columns
# 3. Merges all tables into one master DataFrame
# 4. Engineers key supply chain features
# 5. Uploads clean master table to Google BigQuery
# ============================================================

In [4]:
# Cell 2 — Imports
import pandas as pd
import numpy as np
import os
import pandas_gbq
from google.oauth2 import service_account

print("Libraries loaded ✅")

Libraries loaded ✅


In [5]:
# Cell 3 — Configuration
DATA_PATH  = "../data/"
KEY_PATH   = "bq_key.json"   # update path if different

PROJECT_ID = "supply-chain-analytics-500607"
DATASET_ID = "olist"
TABLE_ID   = "orders_master"

credentials = service_account.Credentials.from_service_account_file(
    KEY_PATH,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Table    : {TABLE_ID}")
print("BigQuery credentials loaded ✅")

Project  : supply-chain-analytics-500607
Dataset  : olist
Table    : orders_master
BigQuery credentials loaded ✅


In [6]:
# Cell 4 — Load all 9 CSVs
orders       = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
items        = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
products     = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
sellers      = pd.read_csv(DATA_PATH + "olist_sellers_dataset.csv")
customers    = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
payments     = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
reviews      = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
geo          = pd.read_csv(DATA_PATH + "olist_geolocation_dataset.csv")
category_map = pd.read_csv(DATA_PATH + "product_category_name_translation.csv")

print("All CSVs loaded ✅")
print(f"orders       : {orders.shape}")
print(f"items        : {items.shape}")
print(f"products     : {products.shape}")
print(f"sellers      : {sellers.shape}")
print(f"customers    : {customers.shape}")
print(f"payments     : {payments.shape}")
print(f"reviews      : {reviews.shape}")
print(f"geo          : {geo.shape}")
print(f"category_map : {category_map.shape}")

All CSVs loaded ✅
orders       : (99441, 8)
items        : (112650, 7)
products     : (32951, 9)
sellers      : (3095, 4)
customers    : (99441, 5)
payments     : (103886, 5)
reviews      : (99224, 7)
geo          : (1000163, 5)
category_map : (71, 2)


In [7]:
# Cell 5 — Parse datetime columns
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Datetime columns parsed ✅")
print(orders[date_cols].dtypes)

Datetime columns parsed ✅
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [8]:
# Cell 6 — Preview orders
print(f"Order statuses:\n{orders['order_status'].value_counts()}")
print(f"\nDate range: {orders['order_purchase_timestamp'].min()} "
      f"→ {orders['order_purchase_timestamp'].max()}")

Order statuses:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Date range: 2016-09-04 21:15:19 → 2018-10-17 17:30:18


In [9]:
# Cell 7 — Merge all tables into master DataFrame
# Step 1 — orders + items
df = orders.merge(items, on="order_id", how="left")
print(f"After orders + items     : {df.shape}")

# Step 2 — + products
df = df.merge(products, on="product_id", how="left")
print(f"After + products         : {df.shape}")

# Step 3 — + category translation
df = df.merge(category_map, on="product_category_name", how="left")
print(f"After + category map     : {df.shape}")

# Step 4 — + sellers
df = df.merge(sellers, on="seller_id", how="left")
print(f"After + sellers          : {df.shape}")

# Step 5 — + customers
df = df.merge(customers, on="customer_id", how="left")
print(f"After + customers        : {df.shape}")

# Step 6 — aggregate payments per order then merge
payments_agg = payments.groupby("order_id").agg(
    total_payment_value   = ("payment_value", "sum"),
    payment_installments  = ("payment_installments", "max"),
    payment_type          = ("payment_type", "first")
).reset_index()
df = df.merge(payments_agg, on="order_id", how="left")
print(f"After + payments         : {df.shape}")

# Step 7 — one review per order (most recent), then merge
reviews_clean = (
    reviews
    .sort_values("review_creation_date", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")
    [["order_id", "review_score", "review_comment_message"]]
)
df = df.merge(reviews_clean, on="order_id", how="left")
print(f"After + reviews          : {df.shape}")
print("\nMaster DataFrame created ✅")

After orders + items     : (113425, 14)
After + products         : (113425, 22)
After + category map     : (113425, 23)
After + sellers          : (113425, 26)
After + customers        : (113425, 30)
After + payments         : (113425, 33)
After + reviews          : (113425, 35)

Master DataFrame created ✅


In [10]:
# Cell 8 — Feature engineering
# 1. Delivery delay in days (positive = late, negative = early)
df["delivery_delay_days"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

# 2. Actual shipping duration (purchase → delivered to customer)
df["shipping_duration_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

# 3. Binary late flag
df["is_late"] = (df["delivery_delay_days"] > 0).astype(int)

# 4. Time features
df["order_month"]      = df["order_purchase_timestamp"].dt.to_period("M").astype(str)
df["order_year"]       = df["order_purchase_timestamp"].dt.year
df["order_month_num"]  = df["order_purchase_timestamp"].dt.month
df["order_dayofweek"]  = df["order_purchase_timestamp"].dt.dayofweek

# 5. Revenue at risk (payment value of late orders only)
df["revenue_at_risk"] = df["total_payment_value"] * df["is_late"]

# 6. Demand concentration score per product
product_velocity = (
    df.groupby("product_id")["order_id"]
    .count()
    .reset_index()
    .rename(columns={"order_id": "order_count"})
)
product_sellers = (
    df.groupby("product_id")["seller_id"]
    .nunique()
    .reset_index()
    .rename(columns={"seller_id": "seller_count"})
)
demand_score = product_velocity.merge(product_sellers, on="product_id")
demand_score["demand_concentration_score"] = (
    demand_score["order_count"] / (demand_score["seller_count"] + 1)
).round(2)
df = df.merge(
    demand_score[["product_id", "demand_concentration_score"]],
    on="product_id",
    how="left"
)

print("Feature engineering complete ✅")
print(df[[
    "delivery_delay_days", "shipping_duration_days",
    "is_late", "revenue_at_risk", "demand_concentration_score"
]].describe().round(2))

Feature engineering complete ✅
       delivery_delay_days  shipping_duration_days    is_late  \
count            110196.00               110196.00  113425.00   
mean                -12.03                   12.01       0.06   
std                  10.16                    9.45       0.24   
min                -147.00                    0.00       0.00   
25%                 -17.00                    6.00       0.00   
50%                 -13.00                   10.00       0.00   
75%                  -7.00                   15.00       0.00   
max                 188.00                  209.00       1.00   

       revenue_at_risk  demand_concentration_score  
count        113422.00                   112650.00  
mean             12.07                       16.33  
std              78.74                       37.75  
min               0.00                        0.50  
25%               0.00                        1.00  
50%               0.00                        3.50  
75%         

In [11]:
# Cell 9 — Clean nulls and filter to delivered orders only
df_clean = df[df["order_status"] == "delivered"].copy()

# Drop rows where key delivery columns are null
df_clean = df_clean.dropna(subset=[
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
])

# Fill remaining nulls
df_clean["review_score"]                    = df_clean["review_score"].fillna(0)
df_clean["review_comment_message"]          = df_clean["review_comment_message"].fillna("")
df_clean["product_category_name_english"]   = df_clean["product_category_name_english"].fillna("unknown")
df_clean["product_weight_g"]                = df_clean["product_weight_g"].fillna(df_clean["product_weight_g"].median())
df_clean["product_length_cm"]               = df_clean["product_length_cm"].fillna(df_clean["product_length_cm"].median())
df_clean["product_height_cm"]               = df_clean["product_height_cm"].fillna(df_clean["product_height_cm"].median())
df_clean["product_width_cm"]                = df_clean["product_width_cm"].fillna(df_clean["product_width_cm"].median())

# Fix column names for BigQuery (no spaces or special characters)
df_clean.columns = df_clean.columns.str.replace(" ", "_").str.lower()

print(f"Clean DataFrame shape    : {df_clean.shape}")
print(f"Delivered orders only    : {len(df_clean)}")
remaining_nulls = df_clean.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
if len(remaining_nulls) == 0:
    print("No nulls remaining ✅")
else:
    print(f"Remaining nulls:\n{remaining_nulls}")

Clean DataFrame shape    : (110189, 44)
Delivered orders only    : 110189
Remaining nulls:
order_approved_at                 15
order_delivered_carrier_date       1
product_category_name           1537
product_name_lenght             1537
product_description_lenght      1537
product_photos_qty              1537
total_payment_value                3
payment_installments               3
payment_type                       3
revenue_at_risk                    3
dtype: int64


In [12]:
# Cell 10 — Preview final clean DataFrame
print("Sample rows:")
df_clean[[
    "order_id", "order_status", "delivery_delay_days",
    "is_late", "revenue_at_risk", "demand_concentration_score",
    "product_category_name_english", "seller_state", "customer_state"
]].head(5)

Sample rows:


,order_id,order_status,delivery_delay_days,is_late,revenue_at_risk,demand_concentration_score,product_category_name_english,seller_state,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,-8.0,0,0.0,2.00,housewares,SP,SP
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,-6.0,0,0.0,21.20,perfumery,SP,BA
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,-18.0,0,0.0,0.75,auto,SP,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,-13.0,0,0.0,2.00,pet_shop,MG,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,-10.0,0,0.0,20.00,stationery,SP,SP


In [13]:
# Cell 11 — Upload to BigQuery
pandas_gbq.to_gbq(
    df_clean,
    destination_table=f"{DATASET_ID}.{TABLE_ID}",
    project_id=PROJECT_ID,
    credentials=credentials,
    if_exists="replace",
    progress_bar=True
)

print(f"\nUploaded {len(df_clean):,} rows to BigQuery ✅")
print(f"Table: {PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")

100%|██████████| 1/1 [00:00<00:00, 6797.90it/s]


Uploaded 110,189 rows to BigQuery ✅
Table: supply-chain-analytics-500607.olist.orders_master


In [14]:
# Cell 12 — Verify upload with a live BigQuery query
from google.cloud import bigquery

client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

result = client.query(f"""
    SELECT
        COUNT(*)                            AS total_rows,
        ROUND(AVG(delivery_delay_days), 2)  AS avg_delay_days,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2) AS late_pct,
        ROUND(SUM(revenue_at_risk), 2)      AS total_revenue_at_risk,
        MIN(order_month)                    AS earliest_month,
        MAX(order_month)                    AS latest_month
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
""").to_dataframe()

print("BigQuery verification ✅")
print(result.to_string())

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


BigQuery verification ✅
   total_rows  avg_delay_days  late_pct  total_revenue_at_risk earliest_month latest_month
0      110189          -12.03      6.59             1368508.71        2016-09      2018-08


In [15]:
# Cell 13 — Summary
print("=" * 55)
print("01_data_ingestion.ipynb — COMPLETE")
print("=" * 55)
print(f"Rows uploaded to BigQuery : {len(df_clean):,}")
print(f"Columns                   : {len(df_clean.columns)}")
print(f"\nKey engineered features:")
print(f"  delivery_delay_days         — delay vs estimated")
print(f"  shipping_duration_days      — actual transit time")
print(f"  is_late                     — binary late flag")
print(f"  revenue_at_risk             — value of late orders")
print(f"  demand_concentration_score  — order velocity / sellers")
print(f"  order_month / order_year    — time features")
print(f"\nBigQuery table ready:")
print(f"  {PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")
print(f"\nNext → run 02_sql_analysis.ipynb")

01_data_ingestion.ipynb — COMPLETE
Rows uploaded to BigQuery : 110,189
Columns                   : 44

Key engineered features:
  delivery_delay_days         — delay vs estimated
  shipping_duration_days      — actual transit time
  is_late                     — binary late flag
  revenue_at_risk             — value of late orders
  demand_concentration_score  — order velocity / sellers
  order_month / order_year    — time features

BigQuery table ready:
  supply-chain-analytics-500607.olist.orders_master

Next → run 02_sql_analysis.ipynb


In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    "bq_key.json",
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
client = bigquery.Client(project="supply-chain-analytics-500607", credentials=credentials)

tables = list(client.list_tables("olist"))
print("BigQuery tables in olist dataset:")
for t in tables:
    print(f"  {t.table_id}")

BigQuery tables in olist dataset:
  delay_by_category
  delay_by_customer_state
  delay_by_seller_state
  demand_concentration
  model_predictions
  monthly_trend
  orders_master
  payment_analysis
  priority_orders
  route_risk
  route_stats
  seller_scorecard
  seller_stats
  sla_analysis
  sla_summary
  stockout_risk_top20
  stockout_scored
  yoy_comparison
